Reading the tables data

## Reading data into dataframe

In [0]:
from pyspark.sql.functions import col, sum, when ,count,countDistinct

In [0]:
date='2026-10-19'

In [0]:
encounter_df=spark.read\
                .option('inferSchema','True')\
                .option('header','True')\
                .csv(f"abfss://bronze@stheathproject.dfs.core.windows.net/Encounters/{date}/Encounters.csv")   
                
organizations_df=spark.read\
                .option('inferSchema','True')\
                .option('header','True')\
                .csv(f"abfss://bronze@stheathproject.dfs.core.windows.net/organizations/{date}/organizations.csv")  
patients_df=spark.read\
                .option('inferSchema','True')\
                .option('header','True')\
                .csv(f"abfss://bronze@stheathproject.dfs.core.windows.net/patients/{date}/patients.csv")  
payers_df=spark.read\
                .option('inferSchema','True')\
                .option('header','True')\
                .csv(f"abfss://bronze@stheathproject.dfs.core.windows.net/payers/{date}/payers.csv")  

procedures_df=spark.read\
                .option('inferSchema','True')\
                .option('header','True')\
                .csv(f"abfss://bronze@stheathproject.dfs.core.windows.net/procedures/{date}/procedures.csv")                                                  


In [0]:
print(encounter_df.columns)
print(organizations_df.columns)
print(patients_df.columns)
print(payers_df.columns)
print(procedures_df.columns)

## Fixed the coulumn names and type

In [0]:
# Fixed the coulumn names

encounter_df=encounter_df.withColumnRenamed('Id','encounter_id').withColumnRenamed('PATIENT','patient_id').withColumnRenamed('ORGANIZATION','ORGANIZATION_id')
organizations_df=organizations_df.withColumnRenamed('Id','ORGANIZATION_id').withColumnRenamed('Name','ORGANIZATION_name')
patients_df=patients_df.withColumnRenamed('Id','patients_id')
payers_df=payers_df.withColumnRenamed('Id','payers_id')
procedures_df=procedures_df.withColumnRenamed('PATIENT','patients_id')




In [0]:
# Function for  all column names to lowercase
def covertToLower(df):
    return  df.toDF(
        *[col.lower() for col in df.columns]
    )

In [0]:
encounter_df=covertToLower(encounter_df)
organizations_df=covertToLower(organizations_df)
patients_df=covertToLower(patients_df)
payers_df=covertToLower(payers_df)
procedures_df=covertToLower(procedures_df)

In [0]:
# Checking duplicates data in patients_df 


duplicate_ids_df = (
    patients_df
    .groupBy("patients_id")
    .count()
    .filter("count > 1")
)

patients_df.join(
    duplicate_ids_df.select('patients_id'),
    on='patients_id',
    how='inner'
).show(10)


In [0]:


encounter_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in encounter_df.columns
]).show(truncate=False)

organizations_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in  organizations_df.columns              
]).show(truncate=False)

procedures_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in procedures_df.columns
]).show(truncate=False)

payers_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in  payers_df.columns              
]).show(truncate=False)

patients_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in  patients_df.columns              
]).show(truncate=False)



In [0]:
encounter_df.filter(
    col("REASONCODE").isNull() &
    col("REASONDESCRIPTION").isNull() &
    col("PAYER").isNull()
).select(
    "Id",
    "REASONCODE",
    "REASONDESCRIPTION",
    "PAYER"
).show(20, truncate=False)